#Ingesta de Banking: bronze

In [0]:
# CONFIGURACIÓN DE PARÁMETROS

dbutils.widgets.text("catalog_name","banking_dev")
dbutils.widgets.text("storage_account","adlsprojectbank12")

#Modo de carga
dbutils.widgets.dropdown("load_mode","FULL",["FULL"])

catalog_name = dbutils.widgets.get("catalog_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
load_mode = dbutils.widgets.get("load_mode").strip().upper()

#Validacion
if not catalog_name or not storage_account:
    raise ValueError("Los parámetros de configuración no pueden estar vacíos")

if load_mode != "FULL":
    raise ValueError(f"Modo de carga no soportado: {load_mode}")

In [0]:
#Mostrar configuración
print("Configuración de ingesta")
print(f"Catálogo       : {catalog_name}")
print(f"Storage Account: {storage_account}")
print(f"Modo de carga  : {load_mode}")

In [0]:
# CONFIGURACIÓN DE RUTAS

raw_base_path = (f"abfss://raw@{storage_account}.dfs.core.windows.net/banking")
bronze_base_path = (f"abfss://bronze@{storage_account}.dfs.core.windows.net")

datasets = ["customers","accounts","cards","transactions","loans","payments"]

source_paths = {name: f"{raw_base_path}/{name}" for name in datasets}
target_tables = {name: f"{catalog_name}.bronze.{name}" for name in datasets}
target_paths = {name: f"{bronze_base_path}/{name}" for name in datasets}

#Mostrar configuración
for name in datasets:
    print(f"{name:<15} | {source_paths[name]}")
    print(f"{'':15} | {target_tables[name]}\n")

In [0]:
# DEFINIR ESQUEMAS
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType, StringType, DateType, TimestampType, DecimalType)

schemas = {
    "customers": StructType([
        StructField("customer_id", LongType(), True),
        StructField("document_type", StringType(), True),
        StructField("document_number", StringType(), True),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("birth_date", DateType(), True),
        StructField("gender", StringType(), True),
        StructField("city", StringType(), True),
        StructField("customer_segment", StringType(), True),
        StructField("registration_date", DateType(), True),
        StructField("customer_status", StringType(), True)
    ]),

    "accounts": StructType([
        StructField("account_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("account_type", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("opening_date", DateType(), True),
        StructField("balance", DecimalType(18, 2), True),
        StructField("account_status", StringType(), True)
    ]),

    "cards": StructType([
        StructField("card_id", LongType(), True),
        StructField("account_id", LongType(), True),
        StructField("card_type", StringType(), True),
        StructField("card_brand", StringType(), True),
        StructField("issue_date", DateType(), True),
        StructField("expiration_date", DateType(), True),
        StructField("card_status", StringType(), True)
    ]),

    "transactions": StructType([
        StructField("transaction_id", LongType(), True),
        StructField("account_id", LongType(), True),
        StructField("card_id", LongType(), True),
        StructField("transaction_date", TimestampType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("merchant_category", StringType(), True),
        StructField("amount", DecimalType(18, 2), True),
        StructField("currency", StringType(), True),
        StructField("transaction_status", StringType(), True),
        StructField("is_fraud", IntegerType(), True)
    ]),

    "loans": StructType([
        StructField("loan_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("loan_type", StringType(), True),
        StructField("principal_amount", DecimalType(18, 2), True),
        StructField("interest_rate", DecimalType(8, 4), True),
        StructField("term_months", IntegerType(), True),
        StructField("disbursement_date", DateType(), True),
        StructField("due_date", DateType(), True),
        StructField("loan_status", StringType(), True)
    ]),

    "payments": StructType([
        StructField("payment_id", LongType(), True),
        StructField("loan_id", LongType(), True),
        StructField("payment_date", DateType(), True),
        StructField("payment_amount", DecimalType(18, 2), True),
        StructField("principal_paid", DecimalType(18, 2), True),
        StructField("interest_paid", DecimalType(18, 2), True),
        StructField("payment_status", StringType(), True)
    ])
}

#Mostrar esquemas
print("Esquemas definidos:")
for name, schema in schemas.items():
    print(f"{name}: {len(schema.fields)} columnas")

In [0]:
# LECTURA DE ARCHIVOS CSV

dataframes = {}

for name in datasets:
    print(f"Leyendo: {name}")

    df = (
        spark.read
        .format("csv")
        .schema(schemas[name])
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss.SSSX") #yyyy-MM-dd HH:mm:ss #yyyy-MM-dd'T'HH:mm:ss.SSSX
        .option("mode", "FAILFAST")
        .load(source_paths[name])
    )

    dataframes[name] = df

    print(f"Columnas: {len(df.columns)}")

print("Lectura de datasets completada")

In [0]:
# INCORPORACION DE INGESTION_DATE

from pyspark.sql import functions as F

for name in datasets:
    df = dataframes[name]

    df = df.withColumn(
        "ingestion_date",
        F.current_timestamp()
    )

    dataframes[name] = df

    print(f"{name}: {len(df.columns)} columnas ,incluyendo ingestion_date\n")

In [0]:
# VALIDAR DATASETS

primary_keys = {
    "customers": "customer_id",
    "accounts": "account_id",
    "cards": "card_id",
    "transactions": "transaction_id",
    "loans": "loan_id",
    "payments": "payment_id"
}

validation_results = {}

for name in datasets:
    df = dataframes[name]
    pk = primary_keys[name]

    total_rows = df.count()
    null_keys = (
        df.filter(F.col(pk).isNull()).count()
    )

    duplicate_keys = (
        df.groupBy(pk)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    validation_results[name] = {
        "total_rows": total_rows,
        "null_keys": null_keys,
        "duplicate_keys": duplicate_keys
    }

    print(
        f"{name:<15} | "
        f"Filas: {total_rows:>8,} | "
        f"PK nulas: {null_keys:>5} | "
        f"PK duplicadas: {duplicate_keys:>5}"
    )

    if null_keys > 0 or duplicate_keys > 0:
        raise Exception(
            f"PK fallida: {name}"
        )

print("Validaciones de PK completas")

In [0]:
# VALIDAR RELACIONES ENTRE TABLAS

customers_df = dataframes["customers"]
accounts_df = dataframes["accounts"]
cards_df = dataframes["cards"]
transactions_df = dataframes["transactions"]
loans_df = dataframes["loans"]
payments_df = dataframes["payments"]

# Accounts a Customers
invalid_accounts = (
    accounts_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    ).count()
)

# Cards a Accounts
invalid_cards = (
    cards_df
    .join(
        accounts_df.select("account_id"),
        on="account_id",
        how="left_anti"
    ).count()
)

# Transactions a Accounts
invalid_transaction_accounts = (
    transactions_df
    .join(
        accounts_df.select("account_id"),
        on="account_id",
        how="left_anti"
    ).count()
)

# Transactions a Cards
invalid_transaction_cards = (
    transactions_df
    .filter(F.col("card_id").isNotNull())
    .join(
        cards_df.select("card_id"),
        on="card_id",
        how="left_anti"
    ).count()
)

# Transactions: tarjeta asociada a la cuenta de la transaccion
invalid_card_account = (
    transactions_df.alias("t")
    .filter(F.col("t.card_id").isNotNull())
    .join(
        cards_df.select(
            "card_id",
            "account_id"
        ).alias("c"),
        on="card_id",
        how="inner"
    )
    .filter(
        F.col("t.account_id") != F.col("c.account_id")
    ).count()
)

# Loans a Customers
invalid_loans = (
    loans_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    ).count()
)

# Payments a Loans
invalid_payments = (
    payments_df
    .join(
        loans_df.select("loan_id"),
        on="loan_id",
        how="left_anti"
    ).count()
)


relationship_results = {
    "Accounts sin cliente": invalid_accounts,
    "Cards sin cuenta": invalid_cards,
    "Transactions sin cuenta": invalid_transaction_accounts,
    "Transactions sin tarjeta": invalid_transaction_cards,
    "Tarjetas asociadas a cuenta incorrecta": invalid_card_account,
    "Loans sin cliente": invalid_loans,
    "Payments sin préstamo": invalid_payments
}

for description, errors in relationship_results.items():
    print(f"{description:<45}: {errors}")

if any(value > 0 for value in relationship_results.values()):
    raise Exception(
        "Se encontraron errores en las relaciones entre datasets"
        "No procede la carga Bronze"
    )

print("Todas las relaciones fueron validadas exitosamente")

In [0]:
# CARGA A TABLAS DELTA

load_results = {}

for name in datasets:

    table_name = target_tables[name]

    print(f"Iniciando carga: {table_name}")

    df = dataframes[name]

    # Respetar exactamente el orden de columnas del destino
    target_columns = spark.table(table_name).columns

    df_to_load = df.select(*target_columns)

    rows_to_load = df_to_load.count()

    (
        df_to_load.write
        .mode("overwrite")
        .insertInto(table_name)
    )

    load_results[name] = {
        "table": table_name,
        "rows_written": rows_to_load,
        "status": "SUCCESS"
    }

    print(
        f"Finalizada: {table_name} | "
        f"Registros escritos: {rows_to_load:,}\n"
    )

print("Proceso de carga Bronze finalizado.")

In [0]:
# VALIDAR CARGA EN TABLAS DELTA

final_results = {}

for name in datasets:
    table_name = target_tables[name]
    expected_rows = load_results[name]["rows_written"]
    actual_rows = (
        spark.table(table_name).count()
    )

    is_valid = expected_rows == actual_rows

    final_results[name] = {
        "expected": expected_rows,
        "actual": actual_rows,
        "status": "SUCCESS" if is_valid else "FAILED"
    }

    print(
        f"{name:<15} | "
        f"Esperados: {expected_rows:>8,} | "
        f"Obtenidos: {actual_rows:>8,} | "
        f"{final_results[name]['status']}\n"
    )

    if not is_valid:
        raise Exception(
            f"Cantidad de registros incorrecta en {table_name}\n"
        )

print("Validación posterior a la carga completada.")

In [0]:
# RESUMEN
total_loaded = sum(
    result["actual"]
    for result in final_results.values()
)

print("=" * 65)
print("RESUMEN DE INGESTA BANKING - BRONZE")
print("=" * 65)

print(f"Catálogo          : {catalog_name}")
print(f"Modo de carga     : {load_mode}")
print(f"Tablas procesadas : {len(final_results)}")
print(f"Total de registros: {total_loaded:,}")

for name, result in final_results.items():

    print(
        f"{name:<15} | "
        f"{result['actual']:>10,} | "
        f"{result['status']}"
    )

print("=" * 65)
print("RESULTADO: INGESTA COMPLETADA CORRECTAMENTE")
print("=" * 65)

###Notebooks de validaciones

In [0]:
from pyspark.sql import functions as F

path_customers = (
    "abfss://raw@adlsprojectbank12.dfs.core.windows.net/"
    "banking/customers"
)

# Listar archivos CSV y sus tamaños
files = dbutils.fs.ls(path_customers)

display([
    (f.name, f.size)
    for f in files
    if f.name.endswith(".csv")
])

In [0]:
for f in files:
    if f.name.endswith(".csv"):
        try:
            df_test = (
                spark.read
                .option("header", "true")
                .option("encoding", "UTF-8")
                .csv(f.path)
            )

            print(
                f"OK | {f.name} | "
                f"Registros: {df_test.count()}"
            )

        except Exception as e:
            print(f"ERROR | {f.name}")
            print(str(e)[:1500])

In [0]:
# Mostrar encabezado real del CSV
for f in files:
    if f.name.endswith(".csv"):
        print(dbutils.fs.head(f.path, 1000))
        break